# Naive Bayes

_Machine Learning_

**Assume features are independent. Multiply their probabilities. Surprisingly good.**

Naive Bayes is a fast, simple classifier built on Bayes' rule.

     It makes one bold ("naive") assumption: features are independent given the class.

---

This notebook is a practice scaffold for the **Naive Bayes** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a labelled dataset and split it

We generate 400 synthetic examples with 6 features (4 of them genuinely informative) so we have a clean, reproducible classification problem. We then split off a quarter of the data as a held-out test set, so accuracy is measured on examples the model never trained on.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=400, n_features=6, n_informative=4,
                           random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit Gaussian Naive Bayes and check accuracy

`GaussianNB` learns, for each class, the mean and variance of every feature, plus the class **priors** `P(y)` (how common each class is). With the naive independence assumption it then multiplies per-feature likelihoods together. We fit on the training split and score on the test split.

In [ ]:
nb_model = GaussianNB().fit(Xtr, ytr)
test_accuracy = round(nb_model.score(Xte, yte), 3)
print("test accuracy:", test_accuracy)
print("class priors P(y):", np.round(nb_model.class_prior_, 3))

### Step 3 — Read off calibrated class probabilities

Naive Bayes doesn't just output a hard label — it gives a probability for each class via Bayes' rule. Here we print the predicted probabilities for the first three test tumors; each row sums to 1 and shows how confident the model is in each class.

In [ ]:
# it outputs calibrated-ish probabilities per class
first_three = Xte[:3]
predicted_probs = nb_model.predict_proba(first_three)
print("first 3 predicted probabilities:")
print(np.round(predicted_probs, 3))

## Visualize the data & results

_Assuming the 30 tumor features are independent given the diagnosis, how confidently does naive Bayes classify real tumors?_

### Step 1 — Train on real breast-cancer tumors

Now we run the same Gaussian Naive Bayes on 569 real breast-cancer scans, each with 30 measured features and a malignant/benign label. We split into train and test, fit the model, and report its accuracy on the held-out tumors.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
X, y = bc.data, bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
nb_model = GaussianNB().fit(Xtr, ytr)
print("accuracy", nb_model.score(Xte, yte))

### Step 2 — Project the 30 features down to 2-D

Thirty features are impossible to plot, so we standardise them and use PCA to compress them into the two directions of greatest spread. This 2-D view lets us *see* how the malignant and benign tumors separate.

In [ ]:
standardized = StandardScaler().fit_transform(X)
P = PCA(n_components=2, random_state=0).fit_transform(standardized)

### Step 3 — Plot the clouds and three confidence scores

The left panel scatters every tumor in PCA space, coloured by true diagnosis. The right panel shows `P(benign)` for three real test tumors — the actual confidence Naive Bayes assigns, which is what drives its predictions.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left — the two tumor clouds in PCA space
for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = P[y == label]
    ax1.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
ax1.set_xlabel("PCA component 1")
ax1.set_ylabel("PCA component 2")
ax1.set_title("Breast-cancer tumors")
ax1.legend()

# Right — predicted P(benign) for three real tumors
benign_probs = nb_model.predict_proba(Xte[:3])[:, 1]
ax2.bar(["tumor 1", "tumor 2", "tumor 3"], benign_probs, color="#ffb454")
ax2.set_title("Predicted P(benign)")

plt.show()